# Fase 3 — Núcleo algorítmico, eficiencia y POO
## Recomendador híbrido de películas TMDB 2010–2025 · Grupo 4

Esta fase convierte los pasos sueltos del notebook de la Fase 2 y los convierten en funciones, clases reutilizables (módulos en `src/`) y mide su costo. Se sigue la estructura del proyecto modelo de la Semana 2 (`Material_Didactico_Semana2`), adaptada al caso TMDB.

**Qué se entrega a la Fase 4:** el conjunto procesado, el pipeline con sus parámetros aprendidos y el esquema final, todo en `data/processed/`.

## Configuración
Única celda a editar. Todo lo demás se construye desde ella.

La configuración identifica la ruta donde se encuentra el conjunto de datos, el esquema (columnas esperadas), la semilla y la fracción que se utilizará para pruebas.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Localiza la raíz del proyecto aunque Jupyter se abra en otra carpeta
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "pipeline.py").exists())
sys.path.insert(0, str(RAIZ))

SEMILLA = 42
RUTA_DATOS = RAIZ / "data" / "raw" / "movies_raw.csv"
DIR_PROCESADO = RAIZ / "data" / "processed"
FRACCION_PRUEBA = 0.20

COLUMNAS_ESPERADAS = [
    "movie_id", "title", "release_date", "status", "budget", "revenue", "runtime",
    "genres", "production_companies", "production_countries", "spoken_languages",
    "original_language", "vote_average", "vote_count", "popularity", "adult",
]
COLUMNAS_NUMERICAS = ["budget", "revenue", "runtime", "vote_average", "vote_count", "popularity"]
COLUMNAS_A_ESCALAR = ["vote_average", "vote_count", "popularity"]
COLUMNA_SESGADA = "popularity"   # log1p antes de escalar: cola larga a la derecha
PUNTUACIONES = ["score_quality_norm", "score_popularity_norm"]

COMPARAR_CON_F2 = True  # Activar para el catálogo oficial con sus exportaciones F2.

## 1. Carga con verificación de esquema
`src/carga.py` falla con un mensaje claro si falta el archivo o una columna declarada.

In [2]:
from src.carga import cargar
df = cargar(RUTA_DATOS, COLUMNAS_ESPERADAS, COLUMNAS_NUMERICAS)
print(f"{df.shape[0]:,} filas x {df.shape[1]} columnas")
df[["title", "genres", "production_companies", "vote_average", "vote_count", "popularity"]].head()

17,978 filas x 16 columnas


,title,genres,production_companies,vote_average,vote_count,popularity
0,Dark Seduction,Drama|Thriller|Romance,Lionsgate,6.799,102,1.6261
1,12,Comedy,NaN,5.000,57,0.3124
2,"Lula, the Son of Brazil",Drama|History,LC Barreto|Filmes do Equador|InterVideo Produc...,5.500,90,1.3337
3,Protect and Serve,Comedy|Crime|Action,M6 Films|Same Player|Pathé|SCOPE Pictures|Webe...,4.500,106,0.5622
4,Re-Cut,Horror|Thriller,Fritz Manger Productions|National Picture Show,6.535,115,0.7341


## 2. Pipeline con clases
Todos los pasos heredan de `Transformador` (contrato `ajustar` / `transformar`, estado encapsulado, copia defensiva). El `Pipeline` los encadena y exige ajustar antes de transformar. Los pasos propios de TMDB (`ConstructorSopa`, `LogNaturalPlus1`, `EscaladorMinMax`) reproducen el Paso 2 y el Paso 5 de la Fase 2.
### 2.1  Resumen F2 decisiones
Resumen de decisiones realizadas para la limpieza del conjunto de datos TMBD en fase 2 y como se implementa en fase 3:
| Decisión F2| Implementación de F3 | Motivo e impacto observado en F2 |
|---|---|---|
| Tratar runtime, budget y revenue | **Transformador.py:**  `CerosComoAusentes` | La columna runtime posee 46 mediciones con valor 0. Las columnas de renueve y budget alcanzan un 62,1% y 69,3% de datos con valor 0. |
| Construir la sopa de metadatos | **Transformador.py:** `ConstructorSopa` |  Las columnas genres y original_language que contengan con datos vacíos usan `Unknown`. Las columnas de production_companies sin datos se dejan vacías para evitar similitud artificial. Se conservan hasta tres productoras sin puntuación ni espacios internos. |
| Representar contenido de la sopa de metadatos| Libreria sklearn `CountVectorizer` con matriz CSR | Cuenta tokens para la similitud coseno. Las matrices dispersas conservan los conteos sin materializar posiciones vacías. |
| Ponderar calidad para calificación de películas | **Transformador.py:** `CalidadPonderada` | Usa la vote_average y el percentil 75 de los vote_count para moderar las notas con menor respaldo. |
| Tratar popularity y escalar puntuaciones |**Transformador.py:** `LogNaturalPlus1`, `EscaladorMinMax` | F2 observó asimetría de ~9,2 en popularidad. `log1p` comprime la cola y conserva el orden. MinMax hace comparables las dos puntuaciones del recomendador. |

### 2.2 Arquitectura del pipeline

- `src/carga.py` convierte la fuente en una tabla y valida columnas y tipos.
- `src/preparacion.py` contiene las reglas de F2 para diagnósticos, limpieza, Weighted Rating y diccionario final. Recibe series o tablas y devuelve resultados reutilizables.
- `src/transformadores.py` encapsula las reglas en clases y conserva los parámetros aprendidos. `Transformador` es la **clase base** y sus subclases **heredan** el ciclo de ajuste y transformación. Usan `aprender` y `aplicar` para implementar su operación.
- `src/pipeline.py` compone pasos ordenados. Los bucles de `ajustar` y `transformar` muestran **polimorfismo** al llamar al mismo método con pasos de distintas clases.
- `src/medicion.py` cronometra funciones y devuelve métricas. `src/utilidades.py` contiene la normalización de nombres usada en F1. Para F3 el mapa de encabezados es explícito en la configuración.

**Principios de POO, patrones de diseño y división funcional**

| Categoría | Concepto | Aplicación en el proyecto | Descripción breve |
|---|---|---|---|
| POO | **Clase base** | `Transformador` en `src/transformadores.py`. | Define la interfaz común con `ajustar()` y `transformar()`. Las subclases implementan `aprender()` y `aplicar()`. |
| POO | **Herencia** | `DuracionPorGenero`, `CerosComoAusentes`, `ConstructorSopa`, `CalidadPonderada`, `LogNaturalPlus1` y `EscaladorMinMax`. | Las subclases reutilizan el ciclo de ajuste y transformación e implementan la operación propia de cada paso. |
| POO | **Polimorfismo** | `Pipeline.ajustar()` y `Pipeline.transformar()`. | El pipeline llama a `ajustar_transformar()` durante el ajuste y a `transformar()` al aplicar el pipeline ya ajustado. Cada paso ejecuta la operación definida por su clase. |
| POO | **Encapsulamiento** | `_ajustado` y `_parametros` en `Transformador`, y `_pasos` en `Pipeline`. | Las clases gestionan estos atributos internos por convención. La propiedad `parametros` devuelve una copia superficial del diccionario, que se consulta al exportar los parámetros en la sección 7. |
| Patrón de diseño | **Método plantilla (Template Method)** | `Transformador.ajustar()` y `Transformador.transformar()`. | La clase base fija la secuencia común y delega el cálculo y la aplicación a `aprender()` y `aplicar()`. |
| Patrón de diseño | **Factory** | `construir_pipeline()` en este notebook. | Cada llamada devuelve un pipeline nuevo con los pasos de TMDB en orden. La comparación con F2 y el ajuste con entrenamiento usan instancias independientes. |
| Diseño estructurado | **Cohesión** | `carga.py`, `preparacion.py`, `transformadores.py`, `pipeline.py` y `medicion.py`. | Cada módulo agrupa una responsabilidad del preprocesamiento o de su evaluación. |
| Diseño estructurado | **Bajo acoplamiento** | `Pipeline` ejecuta los pasos mediante la interfaz de `Transformador`. | Se puede incorporar un transformador compatible en el orden que requieren sus datos de entrada, sin cambiar los bucles del pipeline. |
| Diseño estructurado | **División funcional** | `Pipeline` y funciones como `ceros_a_nan()` y `construir_sopa()` en `src/preparacion.py`. | El trabajo se divide en operaciones reutilizables y el pipeline recorre los pasos con un bucle. |

In [3]:
from src.pipeline import Pipeline
from src.preparacion import diccionario_procesado
from src.transformadores import (CerosComoAusentes, ConstructorSopa, LogNaturalPlus1, EscaladorMinMax,CalidadPonderada,DuracionPorGenero)

COLUMNAS_SALIDA = diccionario_procesado()["columna"].tolist()

def construir_pipeline():
    return Pipeline([
        DuracionPorGenero(),
        CerosComoAusentes("budget", "budget_clean"), 
        CerosComoAusentes("revenue", "revenue_clean"),
        ConstructorSopa(),
        CalidadPonderada(),
        LogNaturalPlus1("popularity", "popularity_log"),
        EscaladorMinMax("weighted_rating", "score_quality_norm"),
        EscaladorMinMax("popularity_log", "score_popularity_norm"),
    ])

display(construir_pipeline().resumen())

,orden,paso,clase
0,1,DuracionPorGenero(runtime),DuracionPorGenero
1,2,CerosComoAusentes(budget),CerosComoAusentes
2,3,CerosComoAusentes(revenue),CerosComoAusentes
3,4,ConstructorSopa(metadata_soup),ConstructorSopa
4,5,CalidadPonderada(vote_average),CalidadPonderada
5,6,LogNaturalPlus1(popularity),LogNaturalPlus1
6,7,EscaladorMinMax(weighted_rating),EscaladorMinMax
7,8,EscaladorMinMax(popularity_log),EscaladorMinMax


## 3. Equivalencia con la Fase 2

Para comprobar que las transformaciones realizadas en la Fase 3 (F3) mantienen los resultados obtenidos en la Fase 2 (F2), se realizan dos comparaciones:

Comparación de datos procesados: Se compara la tabla generada en F3 con el archivo movies_procesado.csv de F2, verificando que coincidan las películas, las columnas y sus valores en el mismo orden. En la columna companies_clean, los valores nulos se reemplazan por textos vacíos para mantener la representación utilizada originalmente. Para los valores numéricos se permite una diferencia mínima de 10⁻¹², debido a posibles diferencias de precisión en los cálculos o al guardar los datos.

Comparación de la matriz de características: Se genera nuevamente la matriz de palabras a partir de metadata_soup, utilizando CountVectorizer, y se compara con la matriz movies_bow_matrix.npz de F2. Se verifica que ambas tengan las mismas dimensiones y que todos sus valores coincidan exactamente.

Objetivo: Verificar que la implementación de F3, basada en clases y transformadores reutilizables, permite obtener los mismos resultados procesados que F2, sin alterar la información utilizada por el recomendador.

In [4]:
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer

if COMPARAR_CON_F2:
    pipeline_catalogo = construir_pipeline().ajustar(df)
    catalogo = pipeline_catalogo.transformar(df)[COLUMNAS_SALIDA]
    referencia = pd.read_csv(DIR_PROCESADO / "movies_procesado.csv")
    referencia["companies_clean"] = referencia["companies_clean"].fillna("")
    pd.testing.assert_frame_equal(
        catalogo, referencia, check_dtype=False, check_exact=False, rtol=1e-12, atol=1e-12
    )
    matriz_catalogo = CountVectorizer().fit_transform(catalogo["metadata_soup"])
    matriz_f2 = sparse.load_npz(DIR_PROCESADO / "movies_bow_matrix.npz")
    assert matriz_catalogo.shape == matriz_f2.shape and (matriz_catalogo != matriz_f2).nnz == 0
    print(f"Equivalencia F2: tabla {catalogo.shape}; matriz CSR {matriz_catalogo.shape}.")
    print(f"Parámetros de calidad F2 reproducidos: {pipeline_catalogo.pasos_ejecutados()[4].parametros}")
else:
    print("Comparación F2 desactivada para una fuente alternativa.")

Equivalencia F2: tabla (17978, 16); matriz CSR (17978, 13766).
Parámetros de calidad F2 reproducidos: {'C': 6.329465569028813, 'm': 427.75}


## 4. Validación: caso normal, casos límite y excepciones
La semilla 42 genera una partición 80/20. Se ajustan medianas, Weighted Rating, escalas y vocabulario en entrenamiento. En prueba se aplican los parámetros aprendidos y los tokens desconocidos quedan fuera del vocabulario. Un extremo nuevo en prueba puede salir de [0, 1] y una fila puede quedar sin coincidencias en el vocabulario. Se registran esos casos.

Los casos límite y los errores esperados se comprueban con datos pequeños. Cada prueba identifica la propiedad que protege.

In [5]:
def debe_fallar(tipo, accion):
    try:
        accion()
    except tipo as e:
        print(f"OK {tipo.__name__}: {e}")
    else:
        raise AssertionError(f"Debía lanzar {tipo.__name__}")


prueba = df.sample(frac=FRACCION_PRUEBA, random_state=SEMILLA)
entrenamiento = df.drop(prueba.index)

if entrenamiento.empty or prueba.empty:
    raise ValueError("La partición requiere filas en entrenamiento y prueba.")

# Aprendemos los parámetros solo con entrenamiento y los reutilizamos en ambas tablas.
pipeline = construir_pipeline().ajustar(entrenamiento)
entrenamiento_final = pipeline.transformar(entrenamiento)[COLUMNAS_SALIDA]
prueba_final = pipeline.transformar(prueba)[COLUMNAS_SALIDA]

# El vocabulario también se aprende en entrenamiento. Prueba usa esas mismas columnas.
vectorizador = CountVectorizer()
matriz_entrenamiento = vectorizador.fit_transform(entrenamiento_final["metadata_soup"])
matriz_prueba = vectorizador.transform(prueba_final["metadata_soup"])

# --- CASO NORMAL: catálogo real ---
# Cada película aparece en una sola partición.
assert set(entrenamiento["movie_id"]).isdisjoint(prueba["movie_id"])

for original, salida, matriz in (
    (entrenamiento, entrenamiento_final, matriz_entrenamiento),
    (prueba, prueba_final, matriz_prueba),
):   
    assert salida["movie_id"].equals(original["movie_id"]) # La transformación conserva las películas y su orden.
    assert matriz.shape == (len(salida), len(vectorizador.vocabulary_)) # Una fila por película y las mismas columnas del vocabulario de entrenamiento.    
    assert salida["metadata_soup"].str.len().gt(0).all()# Las principales variables preparadas cumplen sus condiciones.
    assert salida["runtime_clean"].gt(0).all()
    assert salida["weighted_rating"].between(0, 10).all()
    assert np.isfinite(salida[PUNTUACIONES]).all().all()

# El rango [0, 1] se comprueba en entrenamiento, que fija los extremos de MinMax.
assert entrenamiento_final[PUNTUACIONES].apply(
    lambda columna: columna.between(0, 1)
).all().all()

# En prueba, los valores fuera del rango son un diagnóstico válido.
fuera_de_rango = ~prueba_final[PUNTUACIONES].apply(
    lambda columna: columna.between(0, 1)
)
print("Normal · Particiones, matrices y variables preparadas verificadas.")
print(f"Diagnóstico · Películas de prueba fuera de [0, 1]: {fuera_de_rango.any(axis=1).sum()}")

# --- CASO LÍMITE: una película con metadatos ausentes ---
# Incluimos una duración positiva para que el pipeline pueda aprender la mediana.
minimo = pd.DataFrame({
    "genres": [None],
    "production_companies": [None],
    "original_language": [None],
    "runtime": [90],
    "budget": [0],
    "revenue": [0],
    "vote_average": [7.0],
    "vote_count": [100],
    "popularity": [0.0],
})

salida = construir_pipeline().ajustar(minimo).transformar(minimo)

assert salida["metadata_soup"].iloc[0] == "unknown unknown"
assert salida["vote_average"].iloc[0] == 7.0
assert salida[PUNTUACIONES].iloc[0].tolist() == [0.0, 0.0]
assert salida[["budget_clean", "revenue_clean"]].isna().all().all()
print("Límite · Una fila, metadatos ausentes e importes sin informar.")



debe_fallar(RuntimeError, lambda: construir_pipeline().transformar(entrenamiento))   # sin ajustar
debe_fallar(KeyError, lambda: ConstructorSopa().ajustar(minimo.drop(columns="genres")))
debe_fallar(TypeError, lambda: Pipeline().agregar("no soy un transformador"))
debe_fallar(ValueError, lambda: LogNaturalPlus1("popularity").ajustar_transformar(
    pd.DataFrame({"popularity": [-1.0, 2.0]})))
debe_fallar(FileNotFoundError, lambda: cargar(Path("data/raw/no_existe.csv"), COLUMNAS_ESPERADAS))

Normal · Particiones, matrices y variables preparadas verificadas.
Diagnóstico · Películas de prueba fuera de [0, 1]: 3
Límite · Una fila, metadatos ausentes e importes sin informar.
OK RuntimeError: Pipeline: hay que ajustar antes de transformar.
OK KeyError: "ConstructorSopa(metadata_soup): faltan columnas de origen ['genres']."
OK TypeError: Se esperaba un Transformador y se recibió str.
OK ValueError: LogNaturalPlus1(popularity): log1p no admite valores negativos.
OK FileNotFoundError: No se encontró el archivo: data\raw\no_existe.csv


## 5. Eficiencia: tiempo y memoria
Se compara la construcción de la sopa con un bucle por fila (`iterrows`) frente a la versión por columnas. Se exige el **mismo resultado** antes de comparar y se conserva el menor tiempo de varias repeticiones. El tiempo y la memoria se miden por separado porque `tracemalloc` (memoria) ralentiza la ejecución.

In [6]:
from src.medicion import medir
from src.preparacion import clean_genres, clean_text_feature, imputar_metadatos
import time

def sopa_bucle(df):
    """Construye la sopa fila por fila con las reglas de Fase 2."""
    filas = []
    metadatos = imputar_metadatos(df)

    for _, fila in metadatos.iterrows():
        g = clean_genres(fila["genres"])
        pr = clean_text_feature(fila["production_companies"])
        idi = fila["original_language"].lower()
        filas.append(" ".join(f"{g} {pr} {idi}".split()))

    return pd.Series(filas, index=df.index, name="metadata_soup")


def sopa_columnas(df):
    return ConstructorSopa().ajustar_transformar(df)["metadata_soup"]


def tiempo_minimo(funcion, df, repeticiones=3):
    # tracemalloc ralentiza el código: el tiempo se mide aparte, sin él
    tiempos = []
    for _ in range(repeticiones):
        inicio = time.perf_counter()
        resultado = funcion(df)
        tiempos.append(time.perf_counter() - inicio)
    return resultado, min(tiempos)


r_bucle, t_bucle = tiempo_minimo(sopa_bucle, entrenamiento)
r_cols, t_cols = tiempo_minimo(sopa_columnas, entrenamiento)
pd.testing.assert_series_equal(r_bucle, r_cols, check_dtype=False)   # mismo contenido o la comparación no vale

_, _, m_bucle = medir(sopa_bucle, entrenamiento)
_, _, m_cols = medir(sopa_columnas, entrenamiento)

pd.DataFrame({
    "versión": ["bucle iterrows", "por columnas (clase)"],
    "tiempo_s": [t_bucle, t_cols],
    "memoria_pico_MB": [m_bucle, m_cols],
}).round(3)

,versión,tiempo_s,memoria_pico_MB
0,bucle iterrows,0.418,6.449
1,por columnas (clase),0.086,8.024


In [7]:
relacion = t_bucle / t_cols
print(f"Tiempo: la versión por columnas es {relacion:.1f}x {'más rápida' if relacion >= 1 else 'más lenta'} que el bucle.")
print(f"Memoria pico: {m_bucle:.1f} MB (bucle) vs {m_cols:.1f} MB (columnas).")

Tiempo: la versión por columnas es 4.9x más rápida que el bucle.
Memoria pico: 6.4 MB (bucle) vs 8.0 MB (columnas).


## 6. Carga de un catálogo de películas con otro separador

Para comprobar que el preprocesamiento admite archivos CSV con distintos separadores, se toman diez películas del conjunto de prueba y se guardan temporalmente en un CSV separado por punto y coma. La función `cargar()` detecta el separador y verifica que el archivo contenga las columnas requeridas por el recomendador.

El catálogo cargado se transforma con el pipeline ajustado en la sección 4, reutilizando los parámetros aprendidos en entrenamiento. Luego se comparan las características obtenidas con las de esas mismas películas en `prueba_final`.

La prueba verifica que cambiar el separador del CSV conserva los resultados del preprocesamiento. Para cargar otro catálogo de películas, se puede cambiar la ruta del archivo siempre que las columnas requeridas tengan el mismo significado.

In [8]:
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd

from src.carga import cargar

# Tomamos diez películas del conjunto de prueba.
peliculas = prueba.head(10).copy()

with TemporaryDirectory() as carpeta:
    ruta_csv = Path(carpeta) / "catalogo_peliculas.csv"
    peliculas.to_csv(ruta_csv, sep=";", index=False)

    # cargar() detecta el separador y verifica los encabezados del catálogo.
    catalogo_cargado = cargar(
        ruta_csv,
        columnas_esperadas=COLUMNAS_ESPERADAS,
        columnas_numericas=COLUMNAS_NUMERICAS,
    )

# Aplicamos el pipeline ya ajustado con entrenamiento en el punto 4.
catalogo_preparado = pipeline.transformar(catalogo_cargado)

# Comparamos la identidad, el orden y las características del recomendador.
columnas_recomendador = ["movie_id", "metadata_soup", *PUNTUACIONES]

obtenido = catalogo_preparado[columnas_recomendador].reset_index(drop=True)
esperado = prueba_final.loc[
    peliculas.index, columnas_recomendador
].reset_index(drop=True)

pd.testing.assert_frame_equal(
    obtenido,
    esperado,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

print("Validación correcta: el CSV con punto y coma conserva las características del recomendador.")
display(catalogo_preparado[["title", "metadata_soup", *PUNTUACIONES]])

Validación correcta: el CSV con punto y coma conserva las características del recomendador.


,title,metadata_soup,score_quality_norm,score_popularity_norm
0,Our Brand Is Crisis,comedy drama participant smokehousepictures en,0.417603,0.232256
1,Two Lives,drama thriller helgelandfilm zinnoberfilmundfe...,0.490215,0.130392
2,Hercules Reborn,adventure action theasylum en,0.402468,0.074180
3,Dollface,horror horsecreekproductions en,0.456209,0.065580
4,Fragtime,animation romance science fiction comedy tears...,0.475403,0.134606
5,Shazam! Fury of the Gods,comedy action fantasy thesafrancompany newline...,0.500454,0.450512
6,Exhuma,mystery horror thriller showbox mcmc pinetownp...,0.699705,0.463269
7,BPM (Beats per Minute),drama lesfilmsdepierre page114 mementoproducti...,0.742720,0.198319
8,Exit 8,horror mystery toho aoipro story ja,0.573435,0.575766
9,Foxcatcher,drama thriller annapurnapictures likelystory en,0.550036,0.285495


## 7. Guardar el resultado y su configuración

Se guardan los conjuntos procesados en `entrenamiento_f3.csv` y `prueba_f3.csv`. Cada fila corresponde a una película e incluye su identificador, su título, la sopa de metadatos y las puntuaciones seleccionadas para el recomendador.

El archivo `parametros_f3.json` registra la semilla, la proporción reservada para prueba y los parámetros de cada paso del pipeline. Estos parámetros se aprendieron con el conjunto de entrenamiento y se reutilizaron al transformar el conjunto de prueba.

Finalmente, se vuelve a leer el CSV de entrenamiento y se comprueba que conserva los nombres y el orden de las columnas, así como la cantidad de películas del resultado en memoria.

In [9]:
import json

DIR_PROCESADO.mkdir(parents=True, exist_ok=True)
COLUMNAS_SALIDA = ["movie_id", "title", "metadata_soup", *PUNTUACIONES]

entrenamiento_final[COLUMNAS_SALIDA].to_csv(DIR_PROCESADO / "entrenamiento_f3.csv", index=False)
prueba_final[COLUMNAS_SALIDA].to_csv(DIR_PROCESADO / "prueba_f3.csv", index=False)

parametros = {
    "semilla": SEMILLA,
    "fraccion_prueba": FRACCION_PRUEBA,
    "pasos": {paso.nombre: paso.parametros for paso in pipeline.pasos_ejecutados()},
}
(DIR_PROCESADO / "parametros_f3.json").write_text(
    json.dumps(parametros, indent=2, ensure_ascii=False), encoding="utf-8")

# Comprobación: lo guardado se vuelve a leer sin sorpresas
leido = pd.read_csv(DIR_PROCESADO / "entrenamiento_f3.csv")
assert list(leido.columns) == COLUMNAS_SALIDA and len(leido) == len(entrenamiento_final)
print("Guardado:", sorted(p.name for p in DIR_PROCESADO.glob("*_f3.*")))

Guardado: ['entrenamiento_f3.csv', 'parametros_f3.json', 'prueba_f3.csv']


## Conclusión de la Fase 3

En esta fase convertimos las reglas de preparación de F2 en un pipeline reutilizable. En la comprobación sobre el catálogo completo, el pipeline reprodujo la tabla de 17.978 películas dentro de la tolerancia numérica establecida y generó una matriz dispersa de 17.978 × 13.766 con los mismos conteos de tokens que en F2.

Para preparar los datos de la Fase 4, ajustamos las medianas de duración, los parámetros de la calidad ponderada, las escalas y el vocabulario con el 80 % de entrenamiento, y los reutilizamos en el 20 % de prueba. Las validaciones cubrieron metadatos ausentes, importes sin informar, errores de entrada y la carga de un CSV con punto y coma. En la medición registrada en el cuaderno, construir la sopa por columnas tomó 0,081 s frente a 0,424 s con el bucle por filas, con un pico de memoria de 8,0 MB frente a 6,4 MB.

Los archivos `entrenamiento_f3.csv` y `prueba_f3.csv` entregan la sopa de metadatos y las puntuaciones normalizadas de calidad y popularidad por película. Junto con `parametros_f3.json`, dejan documentada la preparación que sirve de base para construir y evaluar el recomendador híbrido en la Fase 4.